# 专家系统
***

## 1.专家系统概述

**基础概念**
* 专家系统
    * 将人类专家的领域知识转化为形式化的知识库模型
    * 依托基础逻辑推导规则开展推理运算
    * 以此模拟人类专家的推理思路与决策过程
* 专家系统是最早的通用人工智能系统 
    * 专家系统面料的诸多显示困难，推动人们转向基于统计理论的机器学习方法  

**专家系统的组成**

<div align="center">
  <img src="./images/专家系统组成.png" width="500"/>
</div>

## 2. 正向推理（以专家系统动物识别为例）

**专家规则**
* R1: 如果它有羽毛，那么它是鸟类。
* R2: 如果它产乳，那么它是哺乳动物。
* R3: 如果它是鸟类，而且不会飞，那么它是走禽。
* R4: 如果它是鸟类，而且会飞，那么它是飞禽
* R5: 如果它是走禽，而且脖子长，那么它是鸵鸟。
* R6: 如果它是飞禽，而且脖子长，那么它是仙鹤。
* R7: 如果它是哺乳动物，而且以植物为食，那么它是食草动物。
* R8: 如果它是食草动物，而且脖子长，那么它是长劲鹿

**推理过程**
  
<div align="center">
  <img src="./images/推理过程.png" width="500"/>
</div>

**分步解析**
| 步骤 | 已知事实 | 匹配规则 | 推导新事实 | 说明 |
|:------:|:----------:|:----------:|:----------:|:------:|
| 1 | ['会飞行'，'有羽毛','脖子长'] | （['有羽毛'], 鸟类）| 鸟类 | 条件‘有羽毛’满足，推导鸟类 |
| 2 | ['会飞行'，'有羽毛','脖子长','鸟类'] | （['鸟类','会飞'], 飞禽）| 飞禽 | 条件‘鸟类+会飞’满足，推导飞禽 |
| 3 | ['会飞行'，'有羽毛','脖子长','鸟类','飞禽'] | （['飞禽','脖子长'], 仙鹤）| 仙鹤 | 条件‘飞禽+脖子长’满足，推导仙鹤 |
| 4 | 包含仙鹤事实库 | 其余规则均不满足 | 无| 无新事实，推理终止 |


In [12]:
# 定义产生式规则
rules = ['有羽毛 => 鸟类',   
        '产乳 => 哺乳动物',
        '鸟类 and 会飞行 => 飞禽',
        '飞禽 and 脖子长 => 仙鹤',
        '哺乳动物 and 吃草 => 食草动物',
        '食草动物 and 脖子长 => 长颈鹿']

# 解析规则
def parse_rules(rules):
    parsed_rules = []
    for rule in rules:
        conditions, result = rule.split(' => ')   # 以' => '进行分割，前面的是条件，后面的是结果
        conditions = conditions.split(' and ')    # 再以' and '进行分割，
        parsed_rules.append((conditions,result))
    return parsed_rules

parse_rules(rules)  

[(['有羽毛'], '鸟类'),
 (['产乳'], '哺乳动物'),
 (['鸟类', '会飞行'], '飞禽'),
 (['飞禽', '脖子长'], '仙鹤'),
 (['哺乳动物', '吃草'], '食草动物'),
 (['食草动物', '脖子长'], '长颈鹿')]

In [14]:
# 正向推理过程，以规则和事实为输入
def forward_chain(rules, facts):
    has_new_fact = True    # 让while循环至少能进入第一次
    # 循环推理：只要有新事实，就继续匹配规则
    while has_new_fact:
        has_new_fact = False   # 假设本轮没有新事实
        for rule in rules:
            # 检查当前规则的所有条件是否都在已知事实中
            condition_met = all([x in facts for x in rule[0]])   # 对rule[0]中的所有元素，判断其是否在facts中
            if not condition_met:  # 条件不满足，跳过该规则（本条规则的条件不都在事实中，跳过本条规则看下一条），
                continue
            # 判断结论是否已经包含在事实中
            has_new_fact = rule[1] not in facts  # 该条规则的结论不在事实库中为true，视为新事实
            if has_new_fact:
                facts.append(rule[1])  # 将新事实加入事实库
                print(rule[1]) # 打印推导的新事实
                break  #找到新事实后跳出for循环，开始下一次 while循环（避免漏匹配）
                
forward_chain(parse_rules(rules),['会飞行','有羽毛','脖子长'])                

鸟类
飞禽
仙鹤


## 2. 逆向推理（以专家系统动物识别为例）

**专家规则**
* R1: 如果它有羽毛，那么它是鸟类。
* R2: 如果它产乳，那么它是哺乳动物。
* R3: 如果它是鸟类，而且不会飞，那么它是走禽。
* R4: 如果它是鸟类，而且会飞，那么它是飞禽
* R5: 如果它是走禽，而且脖子长，那么它是鸵鸟。
* R6: 如果它是飞禽，而且脖子长，那么它是仙鹤。
* R7: 如果它是哺乳动物，而且以植物为食，那么它是食草动物。
* R8: 如果它是食草动物，而且脖子长，那么它是长劲鹿
  
**推理过程**
<div align="center">
  <img src="./images/逆向推理.png" width="500"/>
</div>

**分步解析**（目标假设：长颈鹿）

| 步骤 | 验证目标 | 匹配规则 | 条件检查 | 操作 |
|:------:|:----------:|:----------:|:----------:|:------:|
| 1 | 长颈鹿|(['食草动物','脖子长']), '长颈鹿') | 条件: 食草动物 (无)、脖子长 (有) | 递归验证 "食草动物" |
| 2 | 食草动物|(['哺乳动物','吃草'], '食草动物') | 条件: 哺乳动物 (无)、吃草 (无) | 递归验证 "哺乳动物" |
| 3 | 哺乳动物|(['产乳'], '哺乳动物') | 条件: 产乳 (无) | 递归验证 "产乳" |
| 4 | 产乳 | 无匹配规则 | -| 打印 "产乳?" (需用户求证) |
| 5 | 回到食草动物 | 仍缺 "吃草" 条件 |- | 递归验证 "吃草" |
| 6 | 吃草 | 无匹配规则 | -| 打印 "吃草?" (需用户求证) |


In [23]:
# 反向推理过程，以规则、事实和假设的目标作为输入
def backward_chain(rules, facts, hypo):
    # 终止条件1:假设已在事实中，无需验证
    if hypo in facts:
        return
    some_rule_applies = False  # 标记是否有规则能够推导该假设
    for rule in rules:
        if rule[1] != hypo:  # 规则结论不等于假设，跳过
            continue 
        some_rule_applies = True   # 找到能推导假设的规则
        # 检查该规则的条件是否都满足
        condition_met = all([x in facts for x in rule[0]])
        if condition_met:
            facts.append(rule[1])
            return
        # 条件不满足，递归验证每个未满足的条件
        for fact in rule[0]:
            if fact in facts:  # 条件已满足，跳过
                continue
            backward_chain(rules, facts, fact)  # 递归验证子条件
    # 终止条件2:无规则能推导假设，需用户求证
    if not some_rule_applies:
        print('{0}?'.format(hypo))

backward_chain(parse_rules(rules), ['脖子长'], '长颈鹿')

产乳?
吃草?


## 4. 专家系统的核心挑战

**知识获取难**
* 领域专家实践成本高昂，难以持续投入知识库构建
* 需知识工程师与领域专家协作完成知识录入，沟通成本高
* 知识库的一致性、完备性难以保障，容易出现歧义与冲突

**系统性能瓶颈**
* 早期系统多基于LISP等语言开发，解释执行模型效率低下
* 面对大规模规则库时，响应速度与运行稳定性无法满足实际需要

**计算复杂度困境**
* 规则验证本质为NP完全问题，随着规则数量增加
* 计算复杂度呈指数级增长，远超实际应用的可接受范围

基于上述问题，专家系统的局限性日益凸显，人们逐渐转向统计理论驱动的方法，机器学习由此逐步兴起并快速发展。